In [ ]:
from pathlib import Path

dataset_path = Path(r"C:\FoodTest1")

print("Running from:", dataset_path.resolve())

valid_exts = {".jpg", ".jpeg", ".png", ".webp"}

# Get all class folders (Asparagus, Carrots, etc.)
class_folders = sorted([f for f in dataset_path.iterdir() if f.is_dir()])

for class_folder in class_folders:
    class_name = class_folder.name.strip().replace(" ", "_")
    
    print(f"\nProcessing folder: {class_name}")
    
    image_files = sorted([
        f for f in class_folder.iterdir()
        if f.is_file() and f.suffix.lower() in valid_exts
    ])
    
    print(f"Found {len(image_files)} images")
    
    temp_files = []
    
    # Step 1: Rename to temp (avoids overwrite issues)
    for i, file_path in enumerate(image_files, start=1):
        temp_name = class_folder / f"temp_{i:03d}{file_path.suffix.lower()}"
        file_path.rename(temp_name)
        temp_files.append(temp_name)
    
    # Step 2: Rename to final format using folder name
    for i, temp_path in enumerate(temp_files, start=1):
        new_name = f"{class_name}_{i:03d}{temp_path.suffix.lower()}"
        temp_path.rename(class_folder / new_name)

print("\nDone.")

In [1]:
from pathlib import Path
import random
import shutil

source_dir = Path(r"C:\FoodTest1")
output_dir = source_dir.parent / "FoodTest1_Split"

if not source_dir.exists():
    raise FileNotFoundError(f"Could not find {source_dir}")

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

random.seed(15)
valid_exts = {".jpg", ".jpeg", ".png", ".webp"}

for split in ["train", "val", "test"]:
    (output_dir / split).mkdir(parents=True, exist_ok=True)

for class_dir in source_dir.iterdir():
    if not class_dir.is_dir():
        continue

    image_files = [f for f in class_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_exts]
    random.shuffle(image_files)

    n = len(image_files)

    n_val = max(1, round(n * val_ratio)) if n >= 10 else 1
    n_test = max(1, round(n * test_ratio)) if n >= 10 else 1
    n_train = n - n_val - n_test

    if n_train < 1:
        continue

    train_files = image_files[:n_train]
    val_files = image_files[n_train:n_train + n_val]
    test_files = image_files[n_train + n_val:]

    for split in ["train", "val", "test"]:
        (output_dir / split / class_dir.name).mkdir(parents=True, exist_ok=True)

    for f in train_files:
        shutil.copy2(f, output_dir / "train" / class_dir.name / f.name)

    for f in val_files:
        shutil.copy2(f, output_dir / "val" / class_dir.name / f.name)

    for f in test_files:
        shutil.copy2(f, output_dir / "test" / class_dir.name / f.name)

    print(f"{class_dir.name}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

print(f"\nDone. Dataset split created at: {output_dir}")

00_Asparagus: train=22, val=5, test=5
01_Carrots: train=22, val=4, test=4
02_Oysters: train=34, val=7, test=7
03_Pork: train=30, val=6, test=6
04_Salmon: train=26, val=6, test=6
05_Zuccini: train=18, val=4, test=4
06_Strawberries: train=35, val=7, test=7
07_Sausages: train=38, val=8, test=8
08_Garlic: train=42, val=9, test=9
09_Ginger: train=26, val=6, test=6
10_Cauliflower: train=16, val=4, test=4
11_Capsicum: train=29, val=6, test=6
12_Pumpkin: train=30, val=7, test=7
13_Rockmelon: train=49, val=10, test=10
14_Watermelon: train=35, val=8, test=8
15_Avocado: train=33, val=7, test=7
16_Tomato: train=50, val=10, test=10
17_Pineapple: train=46, val=10, test=10
18_Pears: train=44, val=10, test=10
19_Apples: train=62, val=13, test=13
20_Peach: train=40, val=8, test=8
21_Trout: train=23, val=5, test=5
22_Snapper: train=39, val=9, test=9
23_Barra: train=21, val=4, test=4
24_Prawns: train=48, val=10, test=10
25_TropicalFish: train=27, val=6, test=6
26_Steak: train=34, val=8, test=8
27_Chicken

In [2]:
from pathlib import Path
from PIL import Image

# -----------------------------
# Paths
# -----------------------------
src_dir = Path(r"C:\FoodTest1\FoodTest1_Split")       # original dataset
dst_dir = Path(r"C:\FoodTest1\FoodTest1_Split_224")   # new resized dataset

valid_exts = {".jpg", ".jpeg", ".png", ".webp"}

print("Starting resize...")

count = 0

# -----------------------------
# Loop through splits
# -----------------------------
for split in ["train", "val", "test"]:
    split_path = src_dir / split

    if not split_path.exists():
        print(f"Skipping {split} (not found)")
        continue

    print(f"Processing: {split}")

    for class_dir in split_path.iterdir():
        if not class_dir.is_dir():
            continue

        # Create output class folder
        out_class_dir = dst_dir / split / class_dir.name
        out_class_dir.mkdir(parents=True, exist_ok=True)

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() not in valid_exts:
                continue

            out_path = out_class_dir / img_path.name

            try:
                with Image.open(img_path) as img:
                    img = img.convert("RGB")
                    img = img.resize((224, 224))
                    img.save(out_path, quality=95)

                count += 1

            except Exception as e:
                print(f"Failed on {img_path}: {e}")

print(f"\nDone. Total images processed: {count}")
print(f"Saved to: {dst_dir}")

Starting resize...
Processing: train
Processing: val
Processing: test

Done. Total images processed: 1992
Saved to: C:\FoodTest1\FoodTest1_Split_224
